# Notebook 06 — Population Genetics: Hardy-Weinberg Equilibrium

**Module:** 05 — Biology Fundamentals  
**Notebook:** 06 of 18  
**Prerequisites:** NB05  
**Time estimate:** 75 minutes

> **Track A critical.** HWE filtering is a standard GWAS QC step. Interviewers ask
> 'What is Hardy-Weinberg and why do we test for it?' — you need a full answer.

---
## Step 1 — Motivation

In every GWAS, you filter out SNPs that deviate from Hardy-Weinberg equilibrium.
This is because HWE violation in controls often signals a genotyping error, not a
real biological signal. Understanding HWE tells you *why* this filter exists and
what it's actually testing.

---
## Step 2 — Intuition

If you put alleles into a 'gene pool' (like a jar of marbles — red for A, blue for a)
and randomly draw two at a time to form offspring genotypes, what frequencies would
you expect? Hardy-Weinberg says: if p = frequency of A and q = frequency of a, then
the expected genotype frequencies are p², 2pq, and q² — every generation, automatically,
as long as the population mates randomly with no other forces acting.

---
## Step 3 — Biological Background

**Hardy-Weinberg equilibrium conditions (all must hold):**
1. Random mating (panmixia)
2. No mutation
3. No gene flow (no immigration/emigration)
4. No genetic drift (effectively infinite population size)
5. No natural selection (all genotypes equally fit)

**Interpretation of HWE deviation in GWAS:**
- HWE violation in **cases** only: could be real disease association
- HWE violation in **controls**: almost always a genotyping artifact → filter the SNP
- Common threshold: p < 1×10⁻⁶ in controls triggers removal

**HWE and allele frequency:**
Given allele frequencies:
- p = frequency of A (dominant allele)
- q = 1 − p = frequency of a (recessive allele)

Expected genotype frequencies under HWE:

| Genotype | Expected frequency |
|----------|-------------------|
| AA | p² |
| Aa | 2pq |
| aa | q² |

Note: p² + 2pq + q² = (p + q)² = 1 ✓

**Estimating allele frequency from genotype counts:**
If you observe N_AA, N_Aa, N_aa individuals:
- p̂ = (2·N_AA + N_Aa) / (2·N)
- q̂ = 1 − p̂
- N = N_AA + N_Aa + N_aa

---
## Step 4 — Mathematical Explanation

**Chi-square test for HWE:**

Null hypothesis: genotype frequencies match HWE expectations given p̂.

$$\chi^2 = \sum_{i} \frac{(O_i - E_i)^2}{E_i}$$

Degrees of freedom = (number of genotype classes) − 1 − (number of estimated parameters)
= 3 − 1 − 1 = **1**

A p-value < 0.05 (or in GWAS, < 10⁻⁶) suggests the locus is out of HWE.

**Derivation (in one step):**
If alleles pair randomly, P(picking A then A) = p × p = p² → that's the frequency of AA
genotypes. For Aa: two ways to get one of each = 2 × p × q = 2pq.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Cell 6.1 — HWE expected genotype frequencies from allele frequency
def hwe_expected(p: float, n_total: int) -> dict:
    """
    Return expected genotype counts under Hardy-Weinberg.
    p: frequency of allele A; q = 1-p
    """
    q = 1.0 - p
    return {
        'AA': p**2 * n_total,
        'Aa': 2*p*q * n_total,
        'aa': q**2 * n_total,
    }

def estimate_allele_freq(n_aa_homo: int, n_het: int, n_aa_homo_rec: int) -> float:
    """Estimate major allele frequency from genotype counts."""
    n_total = n_aa_homo + n_het + n_aa_homo_rec
    return (2 * n_aa_homo + n_het) / (2 * n_total)

# Example: a SNP in 1000 individuals
N_AA, N_Aa, N_aa = 640, 320, 40
N = N_AA + N_Aa + N_aa
p_hat = estimate_allele_freq(N_AA, N_Aa, N_aa)
q_hat = 1 - p_hat

expected = hwe_expected(p_hat, N)
print(f"Observed:  AA={N_AA}, Aa={N_Aa}, aa={N_aa}")
print(f"p̂={p_hat:.3f}, q̂={q_hat:.3f}")
print(f"Expected:  AA={expected['AA']:.1f}, Aa={expected['Aa']:.1f}, aa={expected['aa']:.1f}")

In [ ]:
# Cell 6.2 — Chi-square test for HWE
def hwe_chi_square_test(n_AA: int, n_Aa: int, n_aa: int) -> dict:
    """
    Test whether observed genotype counts are consistent with HWE.
    Returns chi2 statistic, p-value, and a summary.
    """
    N = n_AA + n_Aa + n_aa
    p_hat = (2*n_AA + n_Aa) / (2*N)
    q_hat = 1 - p_hat

    E_AA = p_hat**2 * N
    E_Aa = 2 * p_hat * q_hat * N
    E_aa = q_hat**2 * N

    observed = np.array([n_AA, n_Aa, n_aa], dtype=float)
    expected = np.array([E_AA, E_Aa, E_aa])

    chi2 = np.sum((observed - expected)**2 / expected)
    p_value = stats.chi2.sf(chi2, df=1)  # df=1 for HWE test

    return {'chi2': chi2, 'p_value': p_value, 'p_hat': p_hat,
            'expected': {'AA': E_AA, 'Aa': E_Aa, 'aa': E_aa},
            'in_hwe': p_value >= 0.05}

# SNP 1: In HWE
r1 = hwe_chi_square_test(640, 320, 40)
print("SNP 1 (640 AA, 320 Aa, 40 aa):")
print(f"  χ² = {r1['chi2']:.3f}, p = {r1['p_value']:.4f}, in HWE: {r1['in_hwe']}")

# SNP 2: Out of HWE (excess heterozygotes — common sign of genotyping error)
r2 = hwe_chi_square_test(400, 540, 60)
print("\nSNP 2 (400 AA, 540 Aa, 60 aa):")
print(f"  χ² = {r2['chi2']:.3f}, p = {r2['p_value']:.4e}, in HWE: {r2['in_hwe']}")
print("  → Excess heterozygotes: common sign of strand-alignment errors in genotyping")

In [ ]:
# Cell 6.3 — HWE genotype frequency surface as a function of p
p_values = np.linspace(0, 1, 200)
q_values = 1 - p_values
freq_AA = p_values**2
freq_Aa = 2 * p_values * q_values
freq_aa = q_values**2

print("Maximum heterozygosity occurs at p = q = 0.5:")
idx = np.argmax(freq_Aa)
print(f"  p={p_values[idx]:.2f}: f(AA)={freq_AA[idx]:.2f}, f(Aa)={freq_Aa[idx]:.2f}, f(aa)={freq_aa[idx]:.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — HWE frequency curves + test result bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: HWE genotype frequencies as function of p
ax = axes[0]
ax.plot(p_values, freq_AA, label='f(AA) = p²', color='steelblue', lw=2)
ax.plot(p_values, freq_Aa, label='f(Aa) = 2pq', color='tomato', lw=2)
ax.plot(p_values, freq_aa, label='f(aa) = q²', color='seagreen', lw=2)
ax.set_xlabel('Frequency of allele A (p)')
ax.set_ylabel('Genotype frequency')
ax.set_title('Hardy-Weinberg genotype frequencies\nas function of allele frequency')
ax.legend()
ax.axvline(0.5, color='gray', ls='--', lw=0.8, label='p=0.5 (max heterozyg.)')

# Panel 2: Observed vs expected for two SNPs
ax = axes[1]
genotypes = ['AA', 'Aa', 'aa']
obs1 = [640, 320, 40]
exp1 = [r1['expected'][g] for g in genotypes]
obs2 = [400, 540, 60]
exp2 = [r2['expected'][g] for g in genotypes]

x = np.arange(3)
w = 0.2
ax.bar(x - 1.5*w, obs1, w, label='SNP1 observed', color='steelblue', alpha=0.8)
ax.bar(x - 0.5*w, exp1, w, label='SNP1 expected (HWE)', color='steelblue', alpha=0.4, hatch='//')
ax.bar(x + 0.5*w, obs2, w, label='SNP2 observed', color='tomato', alpha=0.8)
ax.bar(x + 1.5*w, exp2, w, label='SNP2 expected (HWE)', color='tomato', alpha=0.4, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(genotypes)
ax.set_ylabel('Count')
ax.set_title(f'HWE test: SNP1 (p={r1["p_value"]:.3f}) vs SNP2 (p={r2["p_value"]:.2e})')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `hwe_chi_square_test(n_AA, n_Aa, n_aa)` from scratch.
   Test it: if p = 0.8 and N = 500, what genotype counts would exactly satisfy HWE?
   What chi-square statistic do you get? What p-value?
2. A GWAS filters SNPs with HWE p < 10⁻⁶ in controls. Explain in one paragraph
   why HWE deviation in controls is more informative than HWE deviation in cases.
3. Phenylketonuria (PKU) is autosomal recessive with a disease prevalence of ~1/10,000.
   Using HWE, estimate the carrier frequency. (Hint: q² = disease prevalence.)
4. What happens to HWE if a population undergoes a severe bottleneck (very small
   population for a few generations, then expands)? Which assumption of HWE is violated?

---
## Quiz — Active Recall

1. State the Hardy-Weinberg principle in one sentence.
2. What are the five conditions required for HWE to hold?
3. Given p = 0.7, what are the expected frequencies of AA, Aa, and aa under HWE?
4. In a GWAS, why do we filter SNPs that violate HWE in controls but not (necessarily)
   in cases?
5. A population has 900 AA, 90 Aa, and 10 aa. Is this in HWE? Calculate.

---
## Papers Referenced

- Hardy (1908). DOI: 10.1126/science.28.706.49 (Mendelian proportions in a mixed population)

---
## Reflection

**Date completed:** ____________________

> *[If you see a SNP with 100 AA, 800 Aa, 100 aa in a control cohort — what do you
> suspect? Calculate the chi-square test to confirm.]*

---
*Next: `07_genetic_drift_mutation_migration.ipynb`*